In [1]:
%%capture

import warnings
warnings.filterwarnings('ignore')

import calitp_data_analysis.magics

In [2]:
# import shared_utils

In [3]:
import gcsfs as fs
import geopandas as gpd
import numpy as np
import pandas as pd
from calitp_data_analysis import get_fs, utils
from calitp_data_analysis.sql import to_snakecase
# from siuba import *

fs = get_fs()

In [5]:
import altair as alt
import folium
import matplotlib

In [6]:
alt.data_transformers.enable("vegafusion")

DataTransformerRegistry.enable('vegafusion')

In [4]:
# ! pip install vegafusion

In [7]:
# pip install "vegafusion[embed]>=1.5.0"

In [8]:
# ! pip install --upgrade "vegafusion[embed]>=1.5.0" "vegafusion>=1.5.0"

In [9]:
import _replica_utils

In [10]:
from IPython.display import HTML

In [11]:
from calitp_data_analysis import calitp_color_palette as cp

In [12]:
pd.set_option("display.max_columns", None)

In [13]:
gcs_path = "gs://calitp-analytics-data/data-analyses/big_data/STM/"


In [14]:
display(HTML("<h1>Origin-Destination Big Data Analysis: Southern California POIs</h1>"))

In [15]:
display(HTML("This analysis uses Replica Data to identify the top trip start points within a a group of Caltrans Districts and determine what types of trips are occuring when and where."))

In [16]:
# shape_data_name = "origins/replica-stm_origin_la-03_19_26-origin_layer.zip"

# origins_name = "replica-stm_origin_la-03_18_26-trips_dataset.zip"


shape_data_name = "origins/California_hex7_layer.zip"
origins_name = "D7_8_11_12_replica-stm_regional_travel-06_25_26-trips_dataset.zip"

In [17]:
from calitp_data_analysis import utils


In [18]:
# origins = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, file_type="")

In [19]:
# origins.sample()

In [20]:
with get_fs().open(f"{gcs_path}{shape_data_name}") as f:
        shps = to_snakecase(gpd.read_file(f))

In [21]:
shps.sample()

,name,id,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
4935,8729a46e3ffffff,608718528912031700,33.367,-117.0954,2.2556,325,144.0879,"POLYGON ((-117.08719 33.35551, -117.10334 33.3..."


In [22]:
len(shps)

34623

In [23]:
# shps.explore(column="geoname", tooltip=['geoname', 'customgeo', 'centrlat', 'centrlon', 'sqmi', 'origin',
#        'orig_sqmi'])

In [24]:
df = to_snakecase( pd.read_csv(f"{gcs_path}{origins_name}"))  

In [25]:
df.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng
57818,18377865769461856264,"3 (Tract 2676, Los Angeles, CA)","2676 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 17.02, San Bernardino, CA)","17.02 (San Bernardino, CA)","San Bernardino County, CA",California,private_auto,home,work,17:05:00,18:53:54,108,49.6,unknown_vehicle_type,other_non_bev,NaN,NaN,NaN,non_retail_attraction,non_retail_attraction,multi_family,multi_family,12094430424135468792,5903706709779417367,46.0,male,hispanic_or_latino_origin,employed,in_person,12560.0,private_auto,2,27052.0,two,core,naics56,multiple_units,not_attending_school,high_school,renter,spanish,"1 (Tract 17.02, San Bernardino, CA)","17.02 (San Bernardino, CA)","San Bernardino County, CA",California,"3 (Tract 2676, Los Angeles, CA)","2676 (Los Angeles, CA)","Los Angeles County, CA",California,8729a1981ffffff,8729a0ad0ffffff,608718333994336300,60372676003,-118.4622,34.0247,060710017021,6.087183e+17,34.05,-117.6856


columns `origin_custom` & `destination_custom` are for the hex cells, geoname is lost when uploading to Replica to add to the data. check with reps.

In [26]:
df.destination_custom_id.nunique()

6703

In [27]:
df.origin_custom_id.sample()

125619    608718595349807100
Name: origin_custom_id, dtype: int64

In [28]:
df.destination_custom_id.sample()

97365    6.086932e+17
Name: destination_custom_id, dtype: float64

In [29]:
shps[shps['id']==608718594158624800]

,name,id,centrlat,centrlon,sqmi,origin,orig_sqmi,geometry
27,8729a5614ffffff,608718594158624800,33.767,-118.1908,2.232,14442,6470.5376,"POLYGON ((-118.18255 33.75568, -118.19868 33.7..."


In [30]:
# df.origin_custom_id.value_counts()

In [31]:
# df.origin_bgrp_2020.value_counts()

In [32]:
df_origins, df_dests = _replica_utils.read_in_and_prep_replica_data_w_shp(origins_name, shape_data_name, 'origin_custom_id', 'destination_custom_id')

In [33]:
df_origins.sample()

,activity_id,origin_bgrp_2020,origin_trct_2020,origin_cty_2020,origin_st_2020,destination_bgrp_2020,destination_trct_2020,destination_cty_2020,destination_st_2020,primary_mode,trip_purpose,previous_trip_purpose,trip_start_time,trip_end_time,trip_duration_minutes,trip_distance_miles,vehicle_type,vehicle_fuel_type,transit_submode,transit_agency,transit_route,origin_land_use,origin_building_use,destination_land_use,destination_building_use,trip_taker_person_id,trip_taker_household_id,trip_taker_age,trip_taker_sex,trip_taker_race_ethnicity,trip_taker_employment_status,trip_taker_wfh,trip_taker_individual_income,trip_taker_commute_mode,trip_taker_household_size,trip_taker_household_income,trip_taker_available_vehicles,trip_taker_resident_type,trip_taker_industry,trip_taker_building_type,trip_taker_school_grade_attending,trip_taker_education,trip_taker_tenure,trip_taker_language,trip_taker_home_bgrp_2020,trip_taker_home_trct_2020,trip_taker_home_cty_2020,trip_taker_home_st_2020,trip_taker_work_bgrp_2020,trip_taker_work_trct_2020,trip_taker_work_cty_2020,trip_taker_work_st_2020,origin_custom,destination_custom,origin_custom_id,origin_bgrp_fips_2020,origin_custom_lng,origin_custom_lat,destination_bgrp_fips_2020,destination_custom_id,destination_custom_lat,destination_custom_lng,origin_geometry
29174,16064944671622861838,"1 (Tract 2089.03, Los Angeles, CA)","2089.03 (Los Angeles, CA)","Los Angeles County, CA",California,Out of Region,Out of Region,Out of Region,Out of Region,other_travel_mode,social,home,18:48:00,08:03:57,795,1907.3,NaN,unknown_fuel_type,NaN,NaN,NaN,single_family,single_family,retail,retail,5216640664605299457,8933576787198327127,31.0,female,hispanic_or_latino_origin,employed,in_person,59177.0,private_auto,20,191784.0,three_plus,core,naics56,single_family,not_attending_school,some_college,owner,spanish,"1 (Tract 2089.03, Los Angeles, CA)","2089.03 (Los Angeles, CA)","Los Angeles County, CA",California,"1 (Tract 1918.10, Los Angeles, CA)","1918.10 (Los Angeles, CA)","Los Angeles County, CA",California,8729a1d62ffffff,NaN,608718350654111700,60372089031,-118.2688,34.0547,Out of Region,NaN,NaN,NaN,"POLYGON ((-118.26050 34.04335, -118.27667 34.0..."


In [34]:
(df_origins.sample(50)).explore(column="origin_custom_id")

In [35]:
display(HTML("<h2><strong>Trip Counts</strong></h2>"))

In [36]:
display(HTML(f"The number of trips Traveling <strong>From</strong> the top 20 locations is <strong>{len(df_origins)}</strong>"))


In [37]:
assert df_origins.origin_custom_id.nunique() == 50

In [38]:
unique_origin_list = df_origins.origin_custom_id.unique().tolist()

In [39]:
# unique_origin_list

In [40]:
summary = _replica_utils.return_score_summary_single_df(df_origins, unique_origin_list, "origin_geometry", "origin_custom_id")


In [41]:
# summary.sample()

In [42]:
summary.columns = [col.replace('_', ' ').title() for col in summary.columns]


In [43]:
summary_melt =  pd.melt(
        summary,
        id_vars=["Trip Grouping"],
        value_vars=['Trip Grouping', 'Total Trips',
                    'N Auto Trips', 'Pct Auto Trips',
                    'N Tranist Trips', 'Pct Transit Trips',
                    'N Walking Trips', 'Pct Walking Trips',
                    'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
                    'Auto Mean Miles','Auto Median Miles', 'Auto Max Miles',
                    'Transit Mean Minutes','Transit Median Minutes', 'Transit Max Minutes',
                    'Transit Mean Miles','Transit Median Miles', 'Transit Max Miles',
                    'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes',
                    'Walking Mean Miles','Walking Median Miles', 'Walking Max Miles',
                   ],
        var_name="Metric",  # New column for original measurement names
        # value_name="_Value"
)

In [44]:
display(HTML("<h2><strong>Trip Type Summaries</strong></h2>"))

In [45]:
list_of_dfs = []


for origin in unique_origin_list:
    origins_subset = df_origins[df_origins["origin_custom_id"] == origin]

    modes_breakdown_subset = _replica_utils.get_mode_split(origins_subset, "origin_custom_id")

    list_of_dfs.append(modes_breakdown_subset)

modes_breakdown = pd.concat(list_of_dfs, ignore_index=True)

In [46]:
modes_breakdown.columns = [col.replace('_', ' ').title() for col in modes_breakdown.columns]

In [47]:
modes_breakdown.Mode.value_counts()

Mode
other_travel_mode    50
private_auto         49
auto_passenger       48
public_transit       45
biking               44
commercial           40
on_demand_auto       33
walking              21
Name: count, dtype: int64

In [48]:
# len(modes_breakdown)

In [49]:
alt.Chart((modes_breakdown[modes_breakdown['Mode']=="private_auto"])).mark_bar().encode(
    x=alt.X("Trip Grouping:O", title = "Trip Origin"),
    y=alt.Y("Total Trips:Q", title="Total Number of Trips"),
    color=alt.Color("Mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    tooltip = ['Total Trips', 'Mode']
    ).properties(
    width=800,  
    height=300,
    title='Modes by Trip Type')

RuntimeError: The versions of the vegafusion and vegafusion-python-embed packages must match
and must be version 1.5.0 or greater.
Found:
 - vegafusion==2.0.3
 - vegafusion-python-embed==1.6.9


alt.Chart(...)

In [50]:
# alt.Chart(modes_breakdown).mark_bar().encode(
#     x=alt.X('Trip Grouping:O', title ="Trip Origin"),
#     y= alt.Y('Pct Trips:Q', title="Pct of Total Trips"),
#     color=alt.Color('Trip Grouping:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Grouping')),
#     column= alt.Column('Mode:N', title="Mode"),
#     tooltip=['Pct Trips', 'Mode', 'Trip Grouping']
# ).properties(width = 100, height = 400, title = "Modes by Trip Type")

In [ ]:
display(HTML("<h3><strong>Closer Look at Auto, Transit & Walking Trips</strong></h3>"))

In [ ]:
display(HTML("<strong>Tip:</strong> Use the dropdown menu to select different metrics to see on the chart."))

In [ ]:
metrics_list = summary_melt["Metric"].unique().tolist()

metrics_dropdown = alt.binding_select(
        options=metrics_list,
        name="Metrics: ",
    )
    # Column that controls the bar charts
xcol_param = alt.selection_point(
        fields=["Metric"], value=metrics_list[0], bind=metrics_dropdown
    )

chart = (alt.Chart(summary_melt, title="Metric by Trip Types")
        .mark_bar()
        .encode(
            x=alt.X("value"),
            y=alt.Y("Trip Grouping"),
            color=alt.Color("Trip Grouping",
                                  scale=alt.Scale(
                                      range=cp.CALITP_CATEGORY_BRIGHT_COLORS))
        ).properties(width=400, height=350)
    ).add_params(xcol_param).transform_filter(xcol_param)

display(HTML("""
<style>
form.vega-bindings {
  position: absolute;
  left: 0px;
  top: -0px;
}
</style>
"""))

(chart)

In [ ]:
display(HTML("<br>"
            "<br>"))

In [ ]:
summary_long_all_miles = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Miles', 'Auto Median Miles', 'Auto Max Miles', 
        'Transit Mean Miles', 'Transit Median Miles', 'Transit Max Miles',
        'Walking Mean Miles', 'Walking Median Miles', 'Walking Max Miles'],
        var_name="Metric",  # New column for original measurement names
        value_name="Miles")

In [ ]:
summary_long_all_min = pd.melt(
    summary,
    id_vars=["Trip Grouping"],
    value_vars=[
        'Auto Mean Minutes', 'Auto Median Minutes', 'Auto Max Minutes',
        'Transit Mean Minutes', 'Transit Median Minutes', 'Transit Max Minutes',
        'Walking Mean Minutes', 'Walking Median Minutes', 'Walking Max Minutes'],
        var_name="Metric",  # New column for original measurement names
        value_name="Mintues")

In [ ]:
display(HTML("<h3><strong>Trip Length</strong></h3>"))

In [ ]:
alt.Chart(summary_long_all_min).mark_bar().encode(
    x="Mintues:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="Trip Grouping:O",
    facet=alt.Facet('Trip Grouping:O').columns(4),
).properties(title="Travel Length by Trip Type & Mode")

In [ ]:
alt.Chart(summary_long_all_miles).mark_bar().encode(
    x="Miles:Q",
    y="Metric:O",
    color=alt.Color("Metric:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="Trip Grouping:O",
    facet=alt.Facet('Trip Grouping:O').columns(4),
).properties(title="Travel Distance by Trip Type & Mode")

In [ ]:
display(HTML("<h3><strong>Trip Timing</strong></h3>"))

In [ ]:
display(HTML("<strong>Tip:</strong> You can zoom into each graph to better see the timing of the trips by mode"))

In [ ]:
import datetime

In [ ]:
def add_hour_summary_by_mode(df, grouping_col, grouping_list, mode_col): 
    
    time_start = []
    time_end = []
    
    for grouping_value in grouping_list:

        df_subset_start = df[df[grouping_col]== grouping_value]
        df_subset_end = df[df[grouping_col]== grouping_value]

        df_subset_start['trip_start_time'] = pd.to_datetime(df_subset_start['trip_start_time'])
        start_time_bins = df_subset_start.set_index('trip_start_time')
        result_start_time_bins = start_time_bins.groupby([mode_col, grouping_col]).resample('30T').size().reset_index(name='trip_count')

        df_subset_end['trip_start_time'] = pd.to_datetime(df_subset_end['trip_start_time'])
        end_time_bins = df_subset_end.set_index('trip_start_time')
        result_end_time_bins = end_time_bins.groupby([mode_col, grouping_col]).resample('30T').size().reset_index(name='trip_count')

        time_start.append(result_start_time_bins)
        time_end.append(result_end_time_bins)
            

    start_time_all = pd.concat(time_start)
    end_time_all = pd.concat(time_end)

    return start_time_all, end_time_all
       

In [ ]:
start, end = add_hour_summary_by_mode(origins, "origin_bgrp_2020", origin_tracts_list, "primary_mode")

In [ ]:
alt.Chart(start).mark_bar().encode(
    x=alt.X("trip_start_time:T",axis=alt.Axis(format='%H:%M')),
    y=alt.Y("trip_count:Q", title="Number or Trips"),
    color=alt.Color("primary_mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="origin_bgrp_2020:O",
    facet=alt.Facet('origin_bgrp_2020:O').columns(4)).properties(title="Trip Start Time by Primary Mode")

In [ ]:
alt.Chart(end).mark_bar().encode(
    x=alt.X("trip_start_time:T",axis=alt.Axis(format='%H:%M')),
    y=alt.Y("trip_count:Q", title= "Number of Trips"),
    color=alt.Color("primary_mode:N", scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS)),
    # row="origin_bgrp_2020:O",
    facet=alt.Facet('origin_bgrp_2020:O').columns(4)).properties(title="Trip End Time by Primary Mode")

In [ ]:
melt_dfs = []
time_duration_dfs =  []

for origin in origin_tracts_list:
    origins_subset = origins[origins["origin_bgrp_2020"] == origin]

    origins_subset_time_melt, origins_subset_time_melt_duration = _replica_utils.return_time_metrics(origins_subset, "trip_start_time", "trip_end_time")
    
    melt_dfs.append(origins_subset_time_melt)
    time_duration_dfs.append(origins_subset_time_melt_duration)



time_melt = pd.concat(melt_dfs, ignore_index=True)
time_duration_melt = pd.concat(time_duration_dfs, ignore_index=True)


In [ ]:
# alt.Chart(time_melt).mark_circle(size=150).encode(
#     x=alt.X('Time:T',axis=alt.Axis(format='%H:%M')),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Time'],
# ).properties(height=400, width=600, title="Trips Start and End Times")

In [ ]:
# alt.Chart(time_duration_melt).mark_circle(size=150).encode(
#     x=alt.X('Value:Q', title="Minutes"),
#     y='Metric:O',
#     color=alt.Color("Primary Mode"),
#     tooltip = ['Primary Mode', 'Metric', 'Value']
# ).properties(height=400, width=600, title="Trip Duration for Trips")

In [ ]:
n_trips_origin = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_origin = n_trips_origin.set_geometry("geometry")


In [ ]:
n_trips_dest = (origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry"]).agg(
        {"activity_id": "nunique"}).reset_index()
n_trips_dest = n_trips_dest.set_geometry("geometry")


In [ ]:
display(HTML("<h3><strong>Trips Duration</strong></h3>"))

In [ ]:
# origin_tracts_list

In [ ]:
agg_stats_results = []

for tract in origin_tracts_list:
    
    origins_subset = origins[origins["origin_bgrp_2020"] == tract ]
    origins_subset = origins_subset[origins_subset["primary_mode"]!="other_travel_mode"]
    
    k = 1.5
    group_by_column = "primary_mode"
    value_column = "trip_duration_minutes"

    agg_stats = origins_subset.groupby(group_by_column)[value_column].describe()
    
    agg_stats["iqr"] = agg_stats["75%"] - agg_stats["25%"]
    agg_stats["min_"] = agg_stats["25%"] - k * agg_stats["iqr"]
    agg_stats["max_"] = agg_stats["75%"] + k * agg_stats["iqr"]
    data_points = origins_subset[[value_column, group_by_column]].merge(agg_stats.reset_index()[[group_by_column, "min_", "max_"]])
    
    # This will be the lower end of the whisker
    agg_stats["lower"] = (
        data_points[data_points[value_column] >= data_points["min_"]]
        .groupby(group_by_column)[value_column]
        .min()
    )
    # This will be the upper end of the whisker
    agg_stats["upper"] = (
        data_points[data_points[value_column] <= data_points["max_"]]
        .groupby(group_by_column)[value_column]
        .max()
    )
    
    agg_stats = agg_stats.reset_index()

    agg_stats['origin'] = tract

    agg_stats_results.append(agg_stats)

agg_stats_all = pd.concat(agg_stats_results, ignore_index=True)

In [ ]:
# agg_stats_all.sample()

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact


In [ ]:
dropdown = widgets.Dropdown(
    options=origin_tracts_list,
    value=origin_tracts_list[0],
    description='Origin:',
    disabled=False,
)
output = widgets.Output()


In [ ]:
def make_chart(tract):

    df = agg_stats_all[agg_stats_all['origin']==tract]
    
    base = alt.Chart(df).encode(y = alt.Y("primary_mode:N", title="Primary Mode")).properties(title= f"Trip Durations for {tract}")

    rules = base.mark_rule().encode(
        x= alt.X("lower").title("Trip Duration (Minutes)"),
        x2="upper",)
    
    bars = base.mark_bar(size=14).encode(
        x="25%",
        x2="75%",
        color=alt.Color("primary_mode").legend(None),)
    
    ticks = base.mark_tick(color="white", size=14).encode(x="50%")
    
    outliers = base.transform_flatten(flatten=["outliers"]
                                     ).mark_point(style="boxplot-outliers").encode(
        x="outliers:Q",
        color="primary_mode").add_params(xcol_param).transform_filter(xcol_param)

    chart = (rules + bars + ticks).interactive()
   
    return chart

In [ ]:
interact(make_chart, tract=origin_tracts_list);


In [ ]:
### non drop down version
# for tract in origin_tracts_list:

#     agg_stats_all_subset = agg_stats_all[agg_stats_all["origin"] == tract ]


#     base = alt.Chart(agg_stats_all_subset).encode(y = alt.Y("primary_mode:N", title="Primary Mode")).properties(title= f"Trip Durations for {tract}")

#     rules = base.mark_rule().encode(
#         x= alt.X("lower").title("Trip Duration (Minutes)"),
#         x2="upper",)
    
#     bars = base.mark_bar(size=14).encode(
#         x="25%",
#         x2="75%",
#         color=alt.Color("primary_mode").legend(None),)
    
#     ticks = base.mark_tick(color="white", size=14).encode(
#         x="50%"
#     )
    
#     outliers = base.transform_flatten(
#         flatten=["outliers"]
#     ).mark_point(
#         style="boxplot-outliers"
#     ).encode(
#         x="outliers:Q",
#         color="primary_mode",
#     ).add_params(xcol_param).transform_filter(xcol_param)

#     chart = (rules + bars + ticks)

#     display(chart)

In [ ]:
display(HTML("<h2><strong>Trips by Census Block Groups</strong></h2>"))

In [ ]:
n_trips_origin = n_trips_origin.rename(columns={"activity_id":"Number of Trips", "origin_bgrp_2020":"Origin Census BlockGroup"})

n_trips_dest = n_trips_dest.rename(columns={"destination_bgrp_2020":"Destination Census BlockGroup","activity_id":"Number of Trips"})

In [ ]:
display(HTML("<h4>Trips by Origin</h4>"))

In [ ]:
m = n_trips_origin.explore(name="Trips from Origins", column="Number of Trips", 
                cmap="YlOrRd")
display(m)

In [ ]:
# display(HTML("<h4>Trips by Destination</h3>"))

In [ ]:
# n_trips_dest.sample()

In [ ]:
# m = n_trips_dest.explore(name="Trips by Destination", column="Number of Trips", cmap="YlOrRd")
# m

In [ ]:
# n_trips_dest_mode = (
#     origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["destination_bgrp_2020", "geometry", "primary_mode"]).agg(
#         n_trips=("activity_id", "nunique"),
        
#         trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#         trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
#         trip_distance_miles_median=('trip_distance_miles', 'median'),
#         trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
#     #     trip_start_time_mean=('trip_start_time', 'mean'),
#     #     trip_start_time_median=('trip_start_time', 'median'),
        
#     #     trip_end_time_mean=('trip_end_time', 'mean'),
#     #     trip_end_time_median=('trip_end_time', 'median'),
    
#             ).reset_index()
    
# n_trips_dest_mode = n_trips_dest_mode.set_geometry("geometry")

In [ ]:
# n_trips_origin_mode = (
#     origins>>filter(_.primary_mode != "other_travel_mode")).groupby(["origin_bgrp_2020", "geometry", "primary_mode"]).agg(
#         n_trips=("activity_id", "nunique"),
        
#         trip_duration_minutes_median=('trip_duration_minutes', 'median'),
#         trip_duration_minutes_mean=('trip_duration_minutes', 'mean'),
        
#         trip_distance_miles_median=('trip_distance_miles', 'median'),
#         trip_distance_miles_mean=('trip_distance_miles', 'mean'),
        
#     #     trip_start_time_mean=('trip_start_time', 'mean'),
#     #     trip_start_time_median=('trip_start_time', 'median'),
        
#     #     trip_end_time_mean=('trip_end_time', 'mean'),
#     #     trip_end_time_median=('trip_end_time', 'median'),
    
#             ).reset_index()
        
# n_trips_origin_mode = n_trips_origin_mode.set_geometry("geometry")

In [ ]:
# alt.Chart((n_trips_dest_mode>>filter(_.destination_bgrp_2020 == "1 (Tract 3570, Contra Costa, CA)"))).mark_bar().encode(
#     x=alt.X('primary_mode', title ="Trip Mode"),
#     y= alt.Y('n_trips', title="Number of Trips"),
#     # color=alt.Color('destination_bgrp_2020:N', scale=alt.Scale(range=cp.CALITP_CATEGORY_BRIGHT_COLORS), legend=alt.Legend(title='Trip Type')),
# ).properties(width = 500, height = 400, title = "Modes by Trip Type")

In [ ]:
# n_trips_from_cp_mode.sample()

In [ ]:
# unique_modes = origins.primary_mode.unique()

In [ ]:
# unique_modes

In [ ]:
# unique_modes = ['private_auto', 'public_transit', 'walking']

In [ ]:
# display(HTML("<h2>Trips Originating in the Top 20 tracts in the SCAG Area</h2>"))

In [ ]:
# ##hashing out for now for saving
# replica_utils.return_mode_map(n_trips_to_cp_mode, routes, unique_modes, "to")

In [ ]:
# display(HTML("<h4>Trips Destinations of the Top 20 tracts in the SCAG area</h4>"))

In [ ]:
###hashing out for now for saving
# replica_utils.return_mode_map(n_trips_from_cp_mode, routes, unique_modes, "from")

In [ ]:
display(HTML("<h2>Raw Trip Data Sample</h2>"))

In [ ]:
origins.sample(3)